Para esta guía sobre NLP usamos un dataset de Scikit-Learn `fetch_20newsgroups`. Es un conjunto de datos real compuesto por publicaciones de foros de debate organizados por temáticas.  Seleccionaremos dos categorías contrastantes para trabajar sobre un problema de clasificación binaria: `sci.space` (artículos de ciencia y espacio) y `rec.autos` (artículos de automovilismo).

In [4]:
import string
import matplotlib.pyplot as plt
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Descargá las dependencias necesarias de NLTK
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
categorias = ['sci.space', 'rec.autos']
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categorias,
    remove=('headers', 'footers', 'quotes'),
)

df = pd.DataFrame({'texto': newsgroups.data, 'etiqueta': newsgroups.target})
# Filtrá publicaciones vacías
df = df[df['texto'].str.strip() != ''].reset_index(drop=True)
print(f'Total de documentos cargados: {len(df)}')

Total de documentos cargados: 1892


1. Tokenizá y limpiá un texto de muestra: Tomá el primer documento del dataset, convertilo a minúsculas, eliminá los signos de puntuación y filtrá las palabras vacías (stopwords) en inglés usando nltk.

In [6]:
texto_ejemplo = df['texto'].iloc[0]

# Pasar a minúsculas y separar por palabras (tokenización simple)
tokens = texto_ejemplo.lower().split()

# Obtener stopwords en inglés y signos de puntuación
stop_words = set(stopwords.words('english'))
puntuacion = set(string.punctuation)

# Filtrar tokens
tokens_limpios = [
    word.strip(string.punctuation)
    for word in tokens
    if word not in stop_words and word.isalpha()
]

print('Tokens limpios (primeras 10 palabras):', tokens_limpios[:10])

Tokens limpios (primeras 10 palabras): ['seem', 'hardly', 'like', 'groups', 'discuss', 'legitimate', 'power', 'enforce', 'rights', 'derives']


2. Aplicá Stemming y Lematización: Creá instancias de PorterStemmer y WordNetLemmatizer de nltk. Compará cómo transforman las palabras de tu lista de tokens limpios hacia su raíz.

In [7]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stems = [stemmer.stem(palabra) for palabra in tokens_limpios[:8]]
lemas = [lemmatizer.lemmatize(palabra) for palabra in tokens_limpios[:8]]

# Mostrar comparación
df_comparacion = pd.DataFrame(
    {'Original': tokens_limpios[:8], 'Stemming': stems, 'Lematización': lemas}
)
print(df_comparacion)

     Original Stemming Lematización
0        seem     seem         seem
1      hardly   hardli       hardly
2        like     like         like
3      groups    group        group
4     discuss  discuss       discus
5  legitimate  legitim   legitimate
6       power    power        power
7     enforce   enforc      enforce


3. Armá una representación Bag of Words (BoW): Utilizá CountVectorizer sobre los primeros 5 documentos para generar una matriz de frecuencias e imprimí el vocabulario construido.

In [8]:
vectorizer_bow = CountVectorizer(max_features=10)
bow_matrix = vectorizer_bow.fit_transform(df['texto'][:5])

print(
    'Vocabulario de Bag of Words:', vectorizer_bow.get_feature_names_out()
)

Vocabulario de Bag of Words: ['and' 'for' 'in' 'is' 'it' 'not' 'of' 'that' 'the' 'to']


4. Extraé N-Gramos (Unigramas + Bigramas): Configurá CountVectorizer(ngram_range=(1, 2)) para incluir combinaciones de 2 palabras consecutivas y observá el vocabulario generado.

In [9]:
vectorizer_ngrams = CountVectorizer(ngram_range=(1, 2), max_features=10)
ngrams_matrix = vectorizer_ngrams.fit_transform(df['texto'][:5])

print('Vocabulario con Bigramas:', vectorizer_ngrams.get_feature_names_out())

Vocabulario con Bigramas: ['and' 'for' 'in' 'is' 'it' 'not' 'of' 'that' 'the' 'to']


5. Calculá la matriz TF-IDF: Utilizá TfidfVectorizer para ponderar la importancia de las palabras según la frecuencia de término e inversa en documentos ($TF\times IDF$).

In [10]:
vectorizer_tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = vectorizer_tfidf.fit_transform(df['texto'])

print(f'Forma de la matriz TF-IDF: {tfidf_matrix.shape}')

Forma de la matriz TF-IDF: (1892, 1000)


6. Calculá la Similitud Coseno entre documentos: Usá cosine_similarity de sklearn.metrics.pairwise para comparar la representación vectorial $TF\text{-}IDF$ del primer documento con todos los demás documentos del dataset.

In [11]:
# Tomamos el vector del primer documento y lo comparamos contra el resto
similitudes = cosine_similarity(tfidf_matrix[0], tfidf_matrix).flatten()

# Buscá el documento más similar (excluyendo el primero consigo mismo)
indice_mas_similar = similitudes[1:].argmax() + 1

print(f'Índice del documento más similar: {indice_mas_similar}')
print(f'Puntaje de Similitud Coseno: {similitudes[indice_mas_similar]:.4f}')
print(
    f'Etiqueta doc 0: {df["etiqueta"].iloc[0]} | Etiqueta doc similar: {df["etiqueta"].iloc[indice_mas_similar]}'
)

Índice del documento más similar: 83
Puntaje de Similitud Coseno: 0.6936
Etiqueta doc 0: 1 | Etiqueta doc similar: 1


7. Separá los datos en conjunto de entrenamiento y evaluación: Usá train_test_split dividiendo el dataset en 80% entrenamiento y 20% prueba.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['etiqueta'], test_size=0.2, random_state=42
)

8. Construí un Pipeline e integrá Vectorizador + Clasificador: Creá un Pipeline en Scikit-Learn que combine TfidfVectorizer (para convertir texto a vectores) y LogisticRegression como modelo clasificador.

In [13]:
pipeline_clasificador = Pipeline([
    (
        'tfidf',
        TfidfVectorizer(max_features=2000, stop_words='english'),
    ),  # Extracción de features
    (
        'clf',
        LogisticRegression(random_state=42),
    ),  # Modelo de Clasificación binaria[cite: 6]
])

9. Entrená el Pipeline y realizá predicciones: Aplicá .fit() sobre el conjunto de entrenamiento y realizá las predicciones sobre el set de prueba usando .predict.

In [14]:
pipeline_clasificador.fit(X_train, y_train)
y_pred = pipeline_clasificador.predict(X_test)

10. Evaluá las métricas de rendimiento: Imprimí el classification_report para analizar las métricas de Precision, Recall, F1-Score y Accuracy en el problema de clasificación de textos.

In [15]:
nombres_clases = newsgroups.target_names
print(classification_report(y_test, y_pred, target_names=nombres_clases))

              precision    recall  f1-score   support

   rec.autos       0.98      0.91      0.94       197
   sci.space       0.91      0.98      0.94       182

    accuracy                           0.94       379
   macro avg       0.95      0.95      0.94       379
weighted avg       0.95      0.94      0.94       379

